# Kaggle Pipeline — runner

Thin notebook to run the [Kaggle-Pipeline](https://github.com/HkHk2Prod/Kaggle-Pipeline) package on Kaggle.

**Workflow:** run the setup cell, edit the few metaparameters in the config cell, optionally explore the data, then train.

**Internet:** the setup cell *auto-detects* connectivity. With Internet on (Settings → Internet → On) it `pip install`s the package from GitHub; with Internet off it imports the repo from a Kaggle Dataset you've attached (see *Offline use* at the bottom). The same notebook therefore runs unchanged in both Playground-series and code (internet-off) competitions.

In [ ]:
# 1. Make `kaggle_pipeline` importable — automatically, with or without internet.
#
#   Internet ON  -> pip install from GitHub (pin to a tag, e.g. @v0.1.0, for
#                   reproducibility).
#   Internet OFF -> import the repo from a Kaggle Dataset you've attached
#                   (see "Offline use" at the bottom). No edits needed either way.
import importlib.util
import socket
import subprocess
import sys
from pathlib import Path

REPO_URL = "git+https://github.com/HkHk2Prod/Kaggle-Pipeline.git"


def _has_internet(host="github.com", port=443, timeout=5.0):
    """True if a TCP connection to host:port succeeds within `timeout` seconds."""
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False


def _add_offline_package():
    """Put a repo attached as a Kaggle Dataset on sys.path. Returns True on success."""
    root = Path("/kaggle/input")
    if not root.exists():
        return False
    # Find the kaggle_pipeline package itself a few levels deep, so any dataset
    # name / ZIP nesting works (e.g. <ds>/src/kaggle_pipeline or
    # <ds>/Kaggle-Pipeline-main/src/kaggle_pipeline), and add its parent dir.
    for pattern in ("*/kaggle_pipeline", "*/*/kaggle_pipeline", "*/*/*/kaggle_pipeline"):
        for pkg in sorted(root.glob(pattern)):
            if (pkg / "__init__.py").is_file():
                parent = str(pkg.parent)
                if parent not in sys.path:
                    sys.path.insert(0, parent)
                print(f"[setup] offline: imported kaggle_pipeline from {parent}")
                return True
    return False


if importlib.util.find_spec("kaggle_pipeline") is not None:
    print("[setup] kaggle_pipeline already available")
elif _has_internet():
    print("[setup] internet detected -> pip install from GitHub")
    # mistune/nbconvert: Kaggle's preinstalled versions are old enough that
    # Python 3.12 raises SyntaxWarning on their non-raw backslash escapes,
    # spamming every run log. Upgrading to the upstream-fixed releases makes
    # the warnings go away at the source.
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-U",
            REPO_URL,
            "mistune>=3",
            "nbconvert>=7",
        ],
        check=True,
    )
elif not _add_offline_package():
    raise RuntimeError(
        "No internet, and no offline copy of the package was found. Publish this "
        "repo as a Kaggle Dataset (run scripts/publish_dataset.py, or GitHub: "
        "Code -> Download ZIP then create a Dataset) and attach it via Add Input, "
        "then re-run this cell. See 'Offline use' at the bottom of the notebook."
    )

In [ ]:
# 2. The few metaparameters you tune per competition.
#
# Nothing here is strictly required on Kaggle. The data directory is found by
# scanning the attached inputs for the one containing train/test CSVs, and the
# problem-definition fields (target / task / scoring / ...) and CSV filenames are
# autodetected from the data once it loads. Every autodetected value is printed
# as an `[autodetect] ...` line so the run stays reproducible from its log.
#
# Available cfg options
# ---------------------
#   competition       Kaggle competition slug. Optional: only needed to pick the
#                     right input when several attached inputs contain CSVs (or to
#                     derive the Colab data path). Otherwise data_dir is autodetected.
#   target            Target column name(s). None -> the last non-id column of train.
#   id_col            Id column name(s): kept out of the model, reused in the
#                     submission. Default "id".
#   task              "classification" or "regression". None -> inferred from the
#                     target dtype (text / few-valued integers -> classification,
#                     otherwise regression).
#   scoring           CV metric: "roc_auc", "balanced_accuracy" or
#                     "neg_root_mean_squared_error". None -> "roc_auc" for binary
#                     targets, "balanced_accuracy" for multiclass.
#   prediction_aim    Submission format: "category" (hard labels) or "probability".
#                     None -> "probability" when the task is classification.
#   feature_expressions  pandas.eval strings that add new columns; bools -> 0/1.
#   cv_splits         Number of cross-validation folds.
#   run_eda           Render the EDA suite when the Explore cell is run. Training
#                     ignores it. Off by default; set True to render.
#   speed_up          True -> subsample the data for a fast debug run.
#   max_running_time  Stop the search after this many seconds (Kaggle caps at 12h).
#   seed              The single seed for all randomness (CV folds, model
#                     search, ensemble). None (default) = non-reproducible.
#   train_csv / test_csv / sample_csv  CSV filenames inside the data dir. None ->
#                     found by searching for names containing train / test / sample.
#   data_dir / storage_dir   Path overrides. data_dir is required when running
#                     locally; on Kaggle it is autodetected from the attached inputs.
#
# Knobs that tune the evolutionary search itself (models_per_batch,
# search_sample_fraction, onehot_max_cardinality, ensemble sizes, ...) live on
# `KagglePipelineSettings` in the training cell below, not here.
#
# On Kaggle a minimal config can be as short as `Config()`. The autodetected
# problem-definition fields are left as None below: each is inferred from the data
# and printed as an `[autodetect] ...` line.
from kaggle_pipeline import Config

cfg = Config(
    competition=None,  # slug; set only to disambiguate multiple attached inputs
    target=None,  # -> last non-id column of train.csv
    id_col="id",
    task=None,  # -> inferred from the target dtype
    scoring=None,  # -> roc_auc (binary) / balanced_accuracy (multiclass)
    prediction_aim=None,  # -> "probability" for classification
    # Competition-specific pandas.eval columns (not autodetected); fill in per data.
    # Each expression adds one column; boolean results are cast to 0/1. Example:
    #   "is_low   = some_feature < 25",
    #   "is_high  = other_feature > 30",
    feature_expressions=[],
    run_eda=False,  # set True to render EDA when the Explore cell runs
    speed_up=False,  # set True for a fast debug run on a data subset
    max_running_time=43200,
)

## Explore (optional)

Exploratory data analysis is fully decoupled from training — run this cell only when you want to look at the data. It renders metadata, correlation heatmaps and pairwise plots, and does not affect the training run below. Skip it entirely when training.

`analyze` is opt-in: it only renders when `cfg.run_eda` is true, which is set in the config cell above. It is off by default — set `run_eda=True` there to render it, then run the cell below. The training run further down ignores the flag.

In [ ]:
from kaggle_pipeline import analyze

analyze(cfg)

## Train

`KagglePipeline` evolves features and
models in batches under a runtime budget, then ensembles the best. During the
search it trains on a random **subsample** of the rows
(`search_sample_fraction`, default **0.10** = 10%) so it can evaluate far more
candidates per unit time; the ensemble winners are refit on the **full** data at
the end. It reuses the `cfg` above only to find and load the data (and its
autodetected target/task/scoring).

In [ ]:
from kaggle_pipeline.config import resolve_paths
from kaggle_pipeline.data import load_datasets
from kaggle_pipeline.evolution import KagglePipeline, KagglePipelineSettings

# Reuse cfg to locate + load the data (this also autodetects target/task/scoring).
# check_nulls=False: KagglePipeline imputes numerics and treats missing categoricals
# as their own level, so null-bearing data is fine here.
paths = resolve_paths(cfg)
datasets = load_datasets(cfg, paths.data_dir, check_nulls=False)

settings = KagglePipelineSettings(
    max_runtime_seconds=cfg.max_running_time,  # 12h budget; the run stops itself in time
    search_sample_fraction=0.10,  # train on 10% of rows during the search; refit winners on full data
    enable_ensembling=True,
    cv_splits=cfg.cv_splits,
    models_per_batch=32,
    max_active_features=300,
    onehot_max_cardinality=20,  # never one-hot a categorical with more levels than this
    verbosity=3,  # 0 silent .. 4 debug
    seed=cfg.seed,
    state_dir="kagglepipeline_state",
    # Auto-submit from inside run() (which fit() invokes): refit ensemble
    # winners on the FULL data, predict test, and write submission_path -- all
    # within the 12h budget. The runtime carves out
    # `submission_time_reserve_seconds` (30m by default) from the training
    # window so the submission step actually fits.
    make_submission_on_run=True,
    submission_path="submission.csv",
)

pipeline = KagglePipeline(settings)
# fit() does data prep, runs the evolutionary search under the runtime budget,
# ensembles the winners, and (with make_submission_on_run=True above) writes
# the submission -- all in one call.
pipeline.fit(
    datasets.train,
    test_df=datasets.test,
    sample_df=datasets.sample,  # submission matches its columns/format
    target=cfg.target[0] if cfg.target else None,  # None -> autodetected
    task=cfg.task,
    scoring=cfg.scoring,
    prediction_aim=cfg.prediction_aim,  # 'probability' or 'category'
    id_col=cfg.id_col[0] if cfg.id_col else "id",
    feature_expressions=cfg.feature_expressions,  # df.eval columns (no encodings)
    resume=True,  # warm-start from a checkpoint under state_dir, or auto-scan /kaggle/input/* for a previous output
)
print("Wrote", settings.submission_path)

---
## Offline use (no internet)

The setup cell at the top auto-detects connectivity, so **no code change is needed** for internet-off (code) competitions — when there is no connection it imports the package from a Kaggle Dataset instead of pip-installing it. Publish this repository as a Dataset once:

- **Automated:** from a clone of the repo run `python scripts/publish_dataset.py` (needs `pip install kaggle` and a Kaggle API token at `~/.kaggle/kaggle.json`). It creates the dataset the first time and uploads a new version on later runs.
- **Manual:** on GitHub use *Code → Download ZIP*, then create a Kaggle **Dataset** from the ZIP.

Either way, attach the dataset to this notebook via *Add Input*. The setup cell searches `/kaggle/input` (a few levels deep) for the `kaggle_pipeline` package, so the dataset's name and internal nesting don't matter.